# A Claude agent, through the hackathon gateway

This notebook builds a small tool-calling agent on a Claude model hosted in Microsoft Foundry,
reached through the gateway. It is the shortest honest demonstration that a participant key
works for real agent work, not just a single completion.

Nothing here is Azure-aware. The gateway presents the Anthropic Messages API, so this is the
ordinary `anthropic` SDK pointed at a different base URL.

## Before you run it

```bash
pip install anthropic
```

Set the three values from your key card:

```bash
export ANTHROPIC_BASE_URL="https://<your-gateway>.azure-api.net/claude"
export ANTHROPIC_AUTH_TOKEN="<your key>"
export ANTHROPIC_MODEL="<an alias your key grants>"
```

PowerShell uses `$env:ANTHROPIC_BASE_URL = "..."` and so on.

Two things catch people out:

- **`ANTHROPIC_AUTH_TOKEN`, not `ANTHROPIC_API_KEY`.** The first is sent as
  `Authorization: Bearer`, which is what the gateway validates. The second is sent as
  `x-api-key` and returns a bare 401.
- **Use the alias exactly as printed on your card** — for example `sonnet-5`, not `sonnet`.
  Claude Code and some tools treat the bare family names as their own model slots.

## 1. Configuration

Read from the environment. The key is never written into this notebook, so the file stays safe
to share and to commit.

In [1]:
import os

BASE_URL = os.environ.get("ANTHROPIC_BASE_URL")
AUTH_TOKEN = os.environ.get("ANTHROPIC_AUTH_TOKEN")
MODEL = os.environ.get("ANTHROPIC_MODEL")

missing = [n for n, v in {
    "ANTHROPIC_BASE_URL": BASE_URL,
    "ANTHROPIC_AUTH_TOKEN": AUTH_TOKEN,
    "ANTHROPIC_MODEL": MODEL,
}.items() if not v]
if missing:
    raise SystemExit(f"Set these first: {', '.join(missing)}")

print(f"gateway : {BASE_URL}")
print(f"model   : {MODEL}")
print(f"key     : {len(AUTH_TOKEN)} characters (never printed, never written to this notebook)")

gateway : https://apim-hackgwfl4s7jvpxekno.azure-api.net/claude
model   : sonnet-5
key     : 336 characters (never printed, never written to this notebook)


## 2. The client

`auth_token` is what puts the key in the `Authorization` header. Passing it as `api_key` would
send `x-api-key` instead, which this gateway does not read.

In [2]:
from anthropic import Anthropic

client = Anthropic(base_url=BASE_URL, auth_token=AUTH_TOKEN, api_key=None)
print("client ready")

client ready


## 3. Tools

Two tools, deliberately boring and deterministic, so the run is reproducible and any failure is
the agent's rather than the data's.

The Messages API describes a tool with `input_schema` — a JSON Schema object. That differs from
the OpenAI format, which wraps the schema in a `function` object.

In [3]:
import ast
import json
import operator

ORDERS = {
    "A-1001": {"customer": "Ada Lovelace", "total": 129.50, "status": "shipped"},
    "A-1002": {"customer": "Alan Turing", "total": 74.25, "status": "packing"},
}


def get_order_status(order_id: str) -> dict:
    """Look an order up. Unknown ids return an error the model can act on."""
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return {"error": f"No order {order_id}. Known ids: {', '.join(sorted(ORDERS))}"}
    return {"order_id": order_id.strip().upper(), **order}


_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
}


def calculate(expression: str) -> dict:
    """Arithmetic without eval(): walk the AST and allow only numbers and + - * /."""
    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -walk(node.operand)
        raise ValueError("only numbers and + - * / are allowed")

    try:
        return {"expression": expression, "result": walk(ast.parse(expression, mode="eval").body)}
    except Exception as exc:
        return {"error": str(exc)}


TOOLS = [
    {
        "name": "get_order_status",
        "description": "Look up one order by its id. Returns customer, total and status.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "e.g. A-1001"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression. Use this instead of doing sums yourself.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "e.g. 129.50 + 74.25"}},
            "required": ["expression"],
        },
    },
]

IMPLEMENTATIONS = {"get_order_status": get_order_status, "calculate": calculate}
print(f"{len(TOOLS)} tools registered: {', '.join(IMPLEMENTATIONS)}")

2 tools registered: get_order_status, calculate


## 4. The agent loop

The whole of an agent, in one function.

Call the model. If it stops with `tool_use`, run every tool block it asked for, append the
results as a single `user` message, and call again. Repeat until it stops for any other reason.

`max_turns` is a guard rail: without it, a model that keeps asking for tools loops forever and
quietly spends a budget.

In [4]:
def run_agent(question: str, max_turns: int = 6, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": question}]

    for turn in range(1, max_turns + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )

        for block in response.content:
            if block.type == "text" and block.text.strip() and verbose:
                print(f"[turn {turn}] {block.text.strip()}")

        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text").strip()

        messages.append({"role": "assistant", "content": response.content})

        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            result = IMPLEMENTATIONS[block.name](**block.input)
            if verbose:
                print(f"[turn {turn}] {block.name}({json.dumps(block.input)}) -> {json.dumps(result)}")
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result),
            })

        messages.append({"role": "user", "content": results})

    return f"Stopped after {max_turns} turns without a final answer."


print("agent defined")

agent defined


## 5. Run it

The question needs both tools and cannot be answered from the prompt alone: the model has to
look up two orders, then add them.

In [5]:
QUESTION = (
    "Look up orders A-1001 and A-1002. "
    "What is their combined total, and is either one not yet shipped? "
    "Use the calculator for the arithmetic."
)

answer = run_agent(QUESTION)
print("\n--- final answer ---")
print(answer)

[turn 1] get_order_status({"order_id": "A-1001"}) -> {"order_id": "A-1001", "customer": "Ada Lovelace", "total": 129.5, "status": "shipped"}
[turn 1] get_order_status({"order_id": "A-1002"}) -> {"order_id": "A-1002", "customer": "Alan Turing", "total": 74.25, "status": "packing"}


[turn 2] calculate({"expression": "129.5 + 74.25"}) -> {"expression": "129.5 + 74.25", "result": 203.75}


[turn 3] Here's the summary:

- **A-1001** – Ada Lovelace – $129.50 – Status: **shipped**
- **A-1002** – Alan Turing – $74.25 – Status: **packing** (not yet shipped)

**Combined total: $203.75**

Yes — order A-1002 (Alan Turing's order) has not yet shipped; it's currently in the packing stage.

--- final answer ---
Here's the summary:

- **A-1001** – Ada Lovelace – $129.50 – Status: **shipped**
- **A-1002** – Alan Turing – $74.25 – Status: **packing** (not yet shipped)

**Combined total: $203.75**

Yes — order A-1002 (Alan Turing's order) has not yet shipped; it's currently in the packing stage.


## 6. What the gateway added

The same call, with the raw response, so the governance headers are visible. These come from the
gateway, not from the model: a participant can see their own remaining budget without asking an
organiser.

`x-budget-used` is an estimate. On a streamed response it counts the prompt and not the
completion, so it reads low — the cap underneath it is what actually stops a request.

In [6]:
raw = client.messages.with_raw_response.create(
    model=MODEL,
    max_tokens=16,
    messages=[{"role": "user", "content": "Reply with the single word: ok"}],
)

for header in ("x-governed-by", "x-budget-used", "x-budget-total", "x-budget-remaining"):
    value = raw.headers.get(header)
    if value is not None:
        print(f"{header:22} {value}")

print()
print("model replied:", raw.parse().content[0].text.strip())

x-governed-by          foundry-hackathon-gateway
x-budget-used          5483
x-budget-total         200000
x-budget-remaining     197257

model replied: ok


## When it does not work

| What you see | Cause | Fix |
|---|---|---|
| `401` | The key is in `ANTHROPIC_API_KEY`, so it was sent as `x-api-key` | Use `ANTHROPIC_AUTH_TOKEN`, or pass `auth_token=` to the client |
| `401` | The key has expired, or has not started yet | Ask an organiser for a new one; keys are not extended |
| `403 model_not_permitted` | The alias is not on your key, or you shortened it to `sonnet` / `opus` / `haiku` | Use the alias exactly as printed on your card |
| `403 model_not_configured` | The alias is on your key but not pinned on the gateway | Tell an organiser; this is a gateway problem, not a key problem |
| `403 budget_exhausted` | The budget is spent. It is one-time and does not reset | Retrying cannot help. Ask for a new key |
| `429` | Too many requests or tokens per minute | This one is worth retrying; the SDK backs off for you |

Every error arrives in the Anthropic envelope, so `anthropic.APIStatusError` carries the reason
in its message rather than a parse failure.